##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID(s):**     `1527566`


## 1. Supervised model training


In [1]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

# feature lists and file paths
CONTINUOUS_FEATURES = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
CATEGORICAL_FEATURES = [
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country',
]
LABEL_COL = 'income'
CLASSES = ['<=50K', '>50K']
TRAIN_FILE = 'adult_supervised_train.csv'
TEST_FILE = 'adult_test.csv'
UNLABELLED_FILE = 'adult_unlabelled.csv'
UNLABELLED_WITH_LABELS_FILE = 'adult_unlabelled_with_labels.csv'


# data loading and preprocessing functions

def load_dataset(filepath):
    df = pd.read_csv(filepath, na_values=['', '?', 'MISSING'])
    return df


def preprocess(df):
    cleaned_df = df.dropna(subset = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES)  # Removing rows with null values
    cleaned_df = cleaned_df.drop(columns='fnlwgt')  # Removing 'fnlwgt' column as it is not used
    cleaned_df = cleaned_df.reset_index(drop=True)

    return cleaned_df


def separate_features_label(df):
    label_col = df[LABEL_COL]
    features = df.drop(columns=[LABEL_COL])
    
    return (features, label_col)


def split_continuous_categorical(X):
    X_cont = X[CONTINUOUS_FEATURES]
    X_cat = X[CATEGORICAL_FEATURES]

    return (X_cont, X_cat)


def encode_categorical(X_cat, encoder=None):
    if encoder is None:
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        encoded_categorical = encoder.fit_transform(X_cat)
    else:
        encoded_categorical = encoder.transform(X_cat)
        # Replace unseen categories (-1) with 0 so CategoricalNB can handle them
        encoded_categorical[encoded_categorical == -1] = 0

    return (encoded_categorical.astype(int), encoder)


def create_validation_split(X_cont, X_cat, y, val_fraction=0.2, r_state=42):
    X_cont_train, X_cont_val, X_cat_train, X_cat_val, y_train, y_val = train_test_split(X_cont, X_cat, y, test_size=val_fraction, random_state=r_state)
    return (X_cont_train, X_cont_val, X_cat_train, X_cat_val, y_train, y_val)


# model class

class MixedNaiveBayes:

    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.gaussian_nb = GaussianNB()
        self.categorical_nb = CategoricalNB(alpha=self.alpha)


    def fit(self, X_continuous, X_categorical, y):
        self.gaussian_nb.fit(X_continuous, y)
        self.categorical_nb.fit(X_categorical, y)

        return self


    def predict(self, X_continuous, X_categorical):
        log_proba = self.predict_log_proba(X_continuous, X_categorical)
        indices = np.argmax(log_proba, axis=1)

        return self.gaussian_nb.classes_[indices]


    def predict_log_proba(self, X_continuous, X_categorical):
        gaussian_jll = self.gaussian_nb._joint_log_likelihood(X_continuous)
        categorical_jll = self.categorical_nb._joint_log_likelihood(X_categorical)
        # Double counting log P(c)
        combined = gaussian_jll + categorical_jll - self.categorical_nb.class_log_prior_

        return combined


    def get_priors(self):
        return self.gaussian_nb.class_prior_


    def get_gaussian_params(self):
        means = self.gaussian_nb.theta_
        stds = np.sqrt(self.gaussian_nb.var_)

        return {'means': means, 'stds': stds}


    def get_categorical_params(self):
        return self.categorical_nb.feature_log_prob_


# evaluation functions

def compute_confusion_matrix(y_true, y_pred):
    confusion_matrix = {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0}

    for true, pred in zip(y_true, y_pred):
        # Negative class
        if true == CLASSES[0]:
            if pred == true:
                confusion_matrix['TN'] += 1
            else:
                confusion_matrix['FP'] += 1
        # Positive class
        else:
            if pred == true:
                confusion_matrix['TP'] += 1
            else:
                confusion_matrix['FN'] += 1
    
    return confusion_matrix


def compute_metrics(y_true, y_pred):
    confusion_matrix = compute_confusion_matrix(y_true, y_pred)
    TP, FP = confusion_matrix['TP'], confusion_matrix['FP']
    TN, FN = confusion_matrix['TN'], confusion_matrix['FN']

    correct_prediction = 0
    for true, pred in zip(y_true, y_pred):
        correct_prediction += (true == pred)
    accuracy = correct_prediction / len(y_pred) if len(y_pred) > 0 else -1

    # For '>50K' class label
    positive_precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    positive_recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    positive_f1 = (2 * positive_precision * positive_recall) / (positive_precision + positive_recall) if (positive_precision + positive_recall) > 0 else 0
    positive_class_metrics = {'precision': positive_precision, 'recall': positive_recall, 'f1': positive_f1}

    # For '<=50K' class label
    negative_precision = TN / (TN + FN) if (TN + FN) > 0 else 0
    negative_recall = TN / (TN + FP) if (TN + FP) > 0 else 0
    negative_f1 = (2 * negative_precision * negative_recall) / (negative_precision + negative_recall) if (negative_precision + negative_recall) > 0 else 0
    negative_class_metrics = {'precision': negative_precision, 'recall': negative_recall, 'f1': negative_f1}

    metrics = {'accuracy': accuracy, '<=50K': negative_class_metrics, '>50K': positive_class_metrics}
    return metrics


def posterior_ratio(log_proba):
    negative = log_proba[:, 0]  # '<=50K'
    positive = log_proba[:, 1]  # '>50K'
    log_ratio = positive - negative
    
    ratio = np.exp(log_ratio)
    return ratio


def find_high_confidence(X, log_proba, y_true, class_label, n=5):
    ratios = posterior_ratio(log_proba)

    if class_label == CLASSES[1]:  # >50K: high R = confident
        filtered = np.where(ratios > 1)[0]
        sorted_pos = filtered[np.argsort(-ratios[filtered])]
    else:                          # <=50K: low R = confident
        filtered = np.where(ratios < 1)[0]  
        sorted_pos = filtered[np.argsort(ratios[filtered])]

    top = sorted_pos[:n]
    result = X.iloc[top].copy()
    result['log_R'] = log_proba[top, 1] - log_proba[top, 0] 
    result['true_label'] = y_true.iloc[top].values

    return result


def find_borderline(X, log_proba, n=5):
    ratios = posterior_ratio(log_proba)
    distance = np.abs(1 - ratios)
    top = np.argsort(distance)[:n]

    result = X.iloc[top].copy()
    result['R'] = ratios[top]

    return result


# analysis function

def most_predictive_categories(model, feature_names, n=5):
    log_probs = model.get_categorical_params()

    all_ratios = []
    for i, fname in enumerate(feature_names):
        # log_probs[i] shape: (2, n_categories) -- row 0 = <=50K, row 1 = >50K
        log_r = log_probs[i][1, :] - log_probs[i][0, :]
        for v in range(len(log_r)):
            all_ratios.append((fname, v, np.exp(log_r[v])))

    # Sort by ratio descending -- highest R = most predictive of >50K
    sorted_by_r = sorted(all_ratios, key=lambda x: x[2], reverse=True)

    return {
        '>50K': sorted_by_r[:n],
        '<=50K': sorted_by_r[-n:][::-1],  # lowest R = most predictive of <=50K
    }


# Q1: train model and report learned parameters

initial_dataset = load_dataset(TRAIN_FILE)
cleaned_dataset = preprocess(initial_dataset)
X, y = separate_features_label(cleaned_dataset)
X_cont, X_cat = split_continuous_categorical(X)

# encode all training categorical data
encoded_X_cat, encoder = encode_categorical(X_cat)

# 80/20 validation split for Q3
X_cont_train, X_cont_val, X_cat_train, X_cat_val, y_train, y_val = create_validation_split(X_cont, X_cat, y)
encoded_X_cat_train, _ = encode_categorical(X_cat_train, encoder)
encoded_X_cat_val, _ = encode_categorical(X_cat_val, encoder)

# full model for Q2 evaluation, split model for Q1 params and Q3 active learning
full_classifier = MixedNaiveBayes()
full_classifier.fit(X_cont, encoded_X_cat, y)

classifier = MixedNaiveBayes()
classifier.fit(X_cont_train, encoded_X_cat_train, y_train)

# load and encode test data (used in Q4)
testing_dataset = load_dataset(TEST_FILE)
cleaned_test = preprocess(testing_dataset)
test_X, test_y = separate_features_label(cleaned_test)
test_X_cont, test_X_cat = split_continuous_categorical(test_X)
test_encoded_X_cat, _ = encode_categorical(test_X_cat, encoder)

# 1.1 prior probabilities (from classifier trained on 80% split)
priors = classifier.get_priors()
print("P(<=50K) =", round(priors[0], 4))
print("P(>50K)  =", round(priors[1], 4))

# 1.2 continuous feature parameters per class
params = classifier.get_gaussian_params()
print()
for i, feat in enumerate(CONTINUOUS_FEATURES):
    print(f"{feat}: <=50K mean={params['means'][0][i]:.2f} std={params['stds'][0][i]:.2f}, >50K mean={params['means'][1][i]:.2f} std={params['stds'][1][i]:.2f}")

# 1.3 top 5 most predictive category values per class
result = most_predictive_categories(classifier, CATEGORICAL_FEATURES)
for class_label, entries in result.items():
    print(f"\nMost predictive of {class_label}:")
    for fname, v, r in entries:
        feat_idx = CATEGORICAL_FEATURES.index(fname)
        actual_name = encoder.categories_[feat_idx][v]
        print(f"  {fname} = {actual_name}, R = {r:.2f}")

P(<=50K) = 0.7536
P(>50K)  = 0.2464

age: <=50K mean=36.99 std=13.72, >50K mean=43.96 std=10.37
education-num: <=50K mean=9.64 std=2.45, >50K mean=11.60 std=2.37
capital-gain: <=50K mean=151.49 std=968.34, >50K mean=3542.33 std=13394.02
capital-loss: <=50K mean=56.86 std=319.34, >50K mean=211.16 std=616.33
hours-per-week: <=50K mean=39.45 std=11.90, >50K mean=45.66 std=10.42

Most predictive of >50K:
  education = Prof-school, R = 8.06
  education = Doctorate, R = 6.95
  education = Masters, R = 3.48
  native-country = Taiwan, R = 3.47
  workclass = Self-emp-inc, R = 3.25

Most predictive of <=50K:
  relationship = Own-child, R = 0.05
  occupation = Priv-house-serv, R = 0.10
  education = 9th, R = 0.12
  occupation = Other-service, R = 0.14
  relationship = Other-relative, R = 0.15


## 2. Supervised model evaluation

In [2]:
# evaluate full_classifier on training data

y_pred = full_classifier.predict(X_cont, encoded_X_cat)

# 2.1 confusion matrix and per-class metrics

cm = compute_confusion_matrix(y, y_pred)

print("TP:", cm['TP'], " FP:", cm['FP'])
print("FN:", cm['FN'], " TN:", cm['TN'])

m = compute_metrics(y, y_pred)

print("\nAccuracy:", round(m['accuracy'], 4))

print("<=50K  precision:", round(m['<=50K']['precision'], 4),
      " recall:", round(m['<=50K']['recall'], 4),
      " f1:", round(m['<=50K']['f1'], 4))

print(">50K   precision:", round(m['>50K']['precision'], 4),
      " recall:", round(m['>50K']['recall'], 4),
      " f1:", round(m['>50K']['f1'], 4))


# 2.2 check for unseen categories in test data

raw_encoded = encoder.transform(test_X_cat)

unseen_count = 0
for i in range(len(raw_encoded)):
    if -1 in raw_encoded[i]:
        unseen_count += 1

print("\nUnseen category instances:", unseen_count, "/", len(raw_encoded))

for i, feat in enumerate(CATEGORICAL_FEATURES):
    count = 0
    for row in raw_encoded:
        if row[i] == -1:
            count += 1

    if count > 0:
        print(" ", feat, ":", count, "unseen values")


# 2.3 high/low confidence and borderline predictions

log_proba = full_classifier.predict_log_proba(X_cont, encoded_X_cat)

print("\n(a) High confidence >50K:")
high_50k = find_high_confidence(X, log_proba, y, '>50K')
print(high_50k[['age', 'workclass', 'education', 'occupation', 'hours-per-week', 'log_R', 'true_label']])

print("\n(b) High confidence <=50K:")
high_below_50k = find_high_confidence(X, log_proba, y, '<=50K')
print(high_below_50k[['age', 'workclass', 'education', 'occupation', 'hours-per-week', 'log_R', 'true_label']])

print("\n(c) Borderline (R close to 1):")
borderline = find_borderline(X, log_proba)
print(borderline[['age', 'workclass', 'education', 'occupation', 'hours-per-week', 'R']])

TP: 1928  FP: 801
FN: 1779  TN: 10568

Accuracy: 0.8289
<=50K  precision: 0.8559  recall: 0.9295  f1: 0.8912
>50K   precision: 0.7065  recall: 0.5201  f1: 0.5991

Unseen category instances: 1 / 15087
  native-country : 1 unseen values

(a) High confidence >50K:
      age         workclass    education      occupation  hours-per-week  \
9915   52  Self-emp-not-inc  Prof-school  Prof-specialty              45   
1166   32           Private      HS-grad    Craft-repair              40   
2289   42           Private    Bachelors  Prof-specialty              40   
7855   28           Private    Assoc-voc  Prof-specialty              50   
1194   59           Private  Prof-school  Prof-specialty              40   

            log_R true_label  
9915  4788.912948       >50K  
1166  4782.019421       >50K  
2289  4786.271821       >50K  
7855  4784.246751       >50K  
1194  4789.185520       >50K  

(b) High confidence <=50K:
       age workclass  education       occupation  hours-per-week   

/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)
/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)
/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)


## 3. Extending the model with semi-supervised training

In [3]:
# active learning functions

def select_random(n_unlabelled, n=200, random_state=42):
    rng = np.random.default_rng(random_state)
    return rng.choice(n_unlabelled, size=n, replace=False)


def select_uncertain(model, X_cont_unlabelled, X_cat_unlabelled, n=200):
    log_proba = model.predict_log_proba(X_cont_unlabelled, X_cat_unlabelled)
    ratios = posterior_ratio(log_proba)
    distance = np.abs(1 - ratios)
    top = np.argsort(distance)[:n]

    return top


def reveal_labels(indices):
    initial_dataset = load_dataset(UNLABELLED_WITH_LABELS_FILE)
    cleaned_dataset = preprocess(initial_dataset)

    return cleaned_dataset['income'].iloc[indices]


def active_learning_loop(model, X_cont_train, X_cat_train, y_train,
                         X_cont_unlabelled, X_cat_unlabelled,
                         strategy='uncertain', n=200):
    if strategy == 'uncertain':
        indices = select_uncertain(model, X_cont_unlabelled, X_cat_unlabelled, n)
    else:
        indices = select_random(len(X_cont_unlabelled), n)
    
    selected_y = reveal_labels(indices)
    selected_X_cont_unlabelled = np.array(X_cont_unlabelled)[indices]
    selected_X_cat_unlabelled = np.array(X_cat_unlabelled)[indices]

    new_X_cont = np.vstack([X_cont_train, selected_X_cont_unlabelled])
    new_X_cat = np.vstack([X_cat_train, selected_X_cat_unlabelled])
    new_y = np.concatenate([y_train, selected_y])

    new_model = MixedNaiveBayes(alpha=model.alpha)
    new_model.fit(new_X_cont, new_X_cat, new_y)

    return new_model


# load unlabelled data and encode with training encoder

unlabelled_dataset = load_dataset(UNLABELLED_FILE)
cleaned_unlabelled = preprocess(unlabelled_dataset)

X_cont_unlabelled, X_cat_unlabelled = split_continuous_categorical(cleaned_unlabelled)
encoded_X_cat_unlabelled, _ = encode_categorical(X_cat_unlabelled, encoder)


# active learning with both strategies

random_model = active_learning_loop(
    classifier, X_cont_train, encoded_X_cat_train, y_train,
    X_cont_unlabelled, encoded_X_cat_unlabelled, 'random'
)

uncertain_model = active_learning_loop(
    classifier, X_cont_train, encoded_X_cat_train, y_train,
    X_cont_unlabelled, encoded_X_cat_unlabelled, 'uncertain'
)


# sample of the 200 uncertain instances selected

uncertain_indices = select_uncertain(classifier, X_cont_unlabelled, encoded_X_cat_unlabelled)

print("Sample of 200 instances selected via uncertainty sampling:")
print(cleaned_unlabelled.iloc[uncertain_indices].head(10))


# compare all three models on validation set

print()
for name, mdl in [('Supervised', classifier), ('Random (200)', random_model), ('Uncertain (200)', uncertain_model)]:
    acc = compute_metrics(y_val, mdl.predict(X_cont_val, encoded_X_cat_val))['accuracy']
    print(f"{name}: val accuracy = {acc:.4f}")


# vary n to see how sample size affects accuracy

print()
for strategy in ['random', 'uncertain']:
    accs = []

    for n in [50, 100, 200, 300, 500]:
        temp_model = active_learning_loop(
            classifier, X_cont_train, encoded_X_cat_train, y_train,
            X_cont_unlabelled, encoded_X_cat_unlabelled, strategy, n
        )
        acc = compute_metrics(y_val, temp_model.predict(X_cont_val, encoded_X_cat_val))['accuracy']
        accs.append(round(acc, 4))

    print(f"{strategy}: n=50:{accs[0]}, n=100:{accs[1]}, n=200:{accs[2]}, n=300:{accs[3]}, n=500:{accs[4]}")

Sample of 200 instances selected via uncertainty sampling:
       age     workclass  education  education-num      marital-status  \
14178   44       Private  Bachelors             13  Married-civ-spouse   
8086    32       Private  Bachelors             13  Married-civ-spouse   
2604    32       Private  Bachelors             13  Married-civ-spouse   
2765    63  Self-emp-inc  Bachelors             13  Married-civ-spouse   
4088    34       Private  Bachelors             13  Married-civ-spouse   
5271    34       Private  Bachelors             13  Married-civ-spouse   
6146    34       Private  Bachelors             13  Married-civ-spouse   
11417   34       Private  Bachelors             13  Married-civ-spouse   
10209   34       Private  Bachelors             13  Married-civ-spouse   
7788    30       Private  Bachelors             13  Married-civ-spouse   

             occupation relationship   race   sex  capital-gain  capital-loss  \
14178  Transport-moving      Husband  White  

/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)
/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)


random: n=50:0.8316, n=100:0.8316, n=200:0.8322, n=300:0.8319, n=500:0.8322


/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)
/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)
/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)
/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)
/var/folders/ss/srhnv3jd06v1ydpz2pz9krt00000gn/T/ipykernel_84090/1025992932.py:168: RuntimeWarning: overflow encountered in exp
  ratio = np.exp(log_ratio)


uncertain: n=50:0.8326, n=100:0.8326, n=200:0.8326, n=300:0.8329, n=500:0.8329


## 4. Semi-supervised model evaluation

In [4]:
# 4.1 side-by-side test set comparison

for name, mdl in [('Q1 Supervised', full_classifier), ('Q3 Active Learning', uncertain_model)]:
    y_pred_test = mdl.predict(test_X_cont, test_encoded_X_cat)
    m = compute_metrics(test_y, y_pred_test)

    print(f"{name}:")
    print(f"  accuracy={m['accuracy']:.4f}")
    print(f"  <=50K  P={m['<=50K']['precision']:.4f}  R={m['<=50K']['recall']:.4f}  F1={m['<=50K']['f1']:.4f}")
    print(f"  >50K   P={m['>50K']['precision']:.4f}  R={m['>50K']['recall']:.4f}  F1={m['>50K']['f1']:.4f}")
    print()


# 4.2 confidence distribution before vs after

lp_before = full_classifier.predict_log_proba(test_X_cont, test_encoded_X_cat)
lp_after = uncertain_model.predict_log_proba(test_X_cont, test_encoded_X_cat)

log_r_before = lp_before[:, 1] - lp_before[:, 0]
log_r_after = lp_after[:, 1] - lp_after[:, 0]

print("Log R stats (Q1):", "mean=", round(np.mean(log_r_before), 2),
      "median=", round(np.median(log_r_before), 2),
      "std=", round(np.std(log_r_before), 2))

print("Log R stats (Q3):", "mean=", round(np.mean(log_r_after), 2),
      "median=", round(np.median(log_r_after), 2),
      "std=", round(np.std(log_r_after), 2))

# count how many predictions changed between the two models
changed = 0
for i in range(len(log_r_before)):
    before_class = '>50K' if log_r_before[i] > 0 else '<=50K'
    after_class = '>50K' if log_r_after[i] > 0 else '<=50K'
    if before_class != after_class:
        changed += 1

print("Predictions changed:", changed, "/", len(log_r_before))


# 4.3 how much did the learned parameters shift

p_before = full_classifier.get_gaussian_params()
p_after = uncertain_model.get_gaussian_params()

print()
for i, feat in enumerate(CONTINUOUS_FEATURES):
    b0 = p_before['means'][0][i]
    a0 = p_after['means'][0][i]
    b1 = p_before['means'][1][i]
    a1 = p_after['means'][1][i]

    print(f"{feat}: <=50K mean {b0:.2f} -> {a0:.2f} (diff {a0 - b0:.3f}), "
          f">50K mean {b1:.2f} -> {a1:.2f} (diff {a1 - b1:.3f})")

print("\nPriors Q1:", full_classifier.get_priors())
print("Priors Q3:", uncertain_model.get_priors())


# 4.4 check if top predictors changed

for label, mdl in [('Q1', full_classifier), ('Q3', uncertain_model)]:
    r = most_predictive_categories(mdl, CATEGORICAL_FEATURES)

    for cl, entries in r.items():
        names = []
        for fname, v, ratio in entries:
            feat_idx = CATEGORICAL_FEATURES.index(fname)
            names.append(encoder.categories_[feat_idx][v])

        print(f"{label} top {cl}: {names}")

Q1 Supervised:
  accuracy=0.8307
  <=50K  P=0.8545  R=0.9329  F1=0.8920
  >50K   P=0.7240  R=0.5254  F1=0.6089

Q3 Active Learning:
  accuracy=0.8298
  <=50K  P=0.8542  R=0.9318  F1=0.8913
  >50K   P=0.7204  R=0.5251  F1=0.6075

Log R stats (Q1): mean= 21.44 median= -5.59 std= 337.57
Log R stats (Q3): mean= 24.14 median= -5.65 std= 372.12
Predictions changed: 56 / 15087

age: <=50K mean 37.05 -> 37.04 (diff -0.009), >50K mean 43.94 -> 43.86 (diff -0.076)
education-num: <=50K mean 9.63 -> 9.66 (diff 0.032), >50K mean 11.59 -> 11.64 (diff 0.041)
capital-gain: <=50K mean 157.67 -> 153.41 (diff -4.253), >50K mean 3607.15 -> 3413.45 (diff -193.701)
capital-loss: <=50K mean 55.97 -> 56.88 (diff 0.904), >50K mean 202.36 -> 203.35 (diff 0.992)
hours-per-week: <=50K mean 39.43 -> 39.50 (diff 0.069), >50K mean 45.64 -> 45.71 (diff 0.072)

Priors Q1: [0.7541125 0.2458875]
Priors Q3: [0.74836868 0.25163132]
Q1 top >50K: ['Prof-school', 'Doctorate', 'Married-AF-spouse', 'Masters', 'Taiwan']
Q1 top 